In [1]:
import warnings, pickle
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import wilcoxon
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
while not (ROOT/"outputs").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
OUT = ROOT/"outputs"

clean = pd.read_pickle(OUT/"clean.pkl")
RESULTS_PATH = OUT/"results_protocolB.csv"
RESULTS = []

def log_result(model, tier, protocol, features, encoding,
               y_true, y_pred, n_train=None, notes=""):
    row = dict(model=model, tier=tier, protocol=protocol,
               features=features, encoding=encoding,
               rmse=round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 2),
               mae =round(float(mean_absolute_error(y_true, y_pred)), 2),
               r2  =round(float(r2_score(y_true, y_pred)), 4),
               n_train=n_train, n_test=int(len(y_true)), notes=notes)
    RESULTS.append(row)
    pd.DataFrame(RESULTS).to_csv(RESULTS_PATH, index=False)
    print(f"[{tier:12s}|{protocol:14s}|{features:22s}] {model:22s} "
          f"RMSE={row['rmse']:7.2f} MAE={row['mae']:6.2f} R2={row['r2']:7.4f}")
    return row

print("clean:", clean.shape)
print("python:", __import__("sys").executable)

clean: (45657, 16)
python: c:\Users\hp\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
def build_full_features(df):
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        for L in (1,2,3,6,12,24): g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        for W in (6,24):
            g[f"aqi_roll{W}_mean"]=g.aqi_index.shift(1).rolling(W).mean()
            g[f"aqi_roll{W}_max"] =g.aqi_index.shift(1).rolling(W).max()
            g[f"aqi_roll{W}_std"] =g.aqi_index.shift(1).rolling(W).std()
        g["aqi_ewm12"]=g.aqi_index.shift(1).ewm(span=12).mean()
        g["aqi_mom3"]=g.aqi_lag1-g.aqi_lag3
        g["aqi_ratio24"]=g.aqi_lag1/(g["aqi_roll24_mean"]+1e-6)
        t=g.index.get_level_values("dt")
        g["hour"]=t.hour; g["month"]=t.month; g["dow"]=t.dayofweek
        g["is_weekend"]=(t.dayofweek>=5).astype(int)
        g["hour_sin"]=np.sin(2*np.pi*t.hour/24); g["hour_cos"]=np.cos(2*np.pi*t.hour/24)
        g["month_sin"]=np.sin(2*np.pi*t.month/12); g["month_cos"]=np.cos(2*np.pi*t.month/12)
        g["pressure_delta"]=g.pressure_mb.diff()
        g["wind_x_lag1"]=g.windspeed_kph*g.aqi_lag1
        g["target"]=g.aqi_index.shift(-1)
        g = g.drop(columns=["aqi_index","pm2_5","pm10","co","no2"])
        parts.append(g)
    return pd.concat(parts).dropna(subset=["target","aqi_lag24"])

feat = build_full_features(clean)
print("feature frame:", feat.shape)
feat_reset = feat.reset_index()
print(feat_reset.location.value_counts())

feature frame: (45507, 37)
location
Anand Vihar        8445
Connaught Place    8445
Rohini             8445
Dwarka             6724
IGI Airport        6724
Okhla Phase III    6724
Name: count, dtype: int64


In [3]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              VotingRegressor)
from xgboost import XGBRegressor
import lightgbm as lgb

BASE_DROP = ["target","date_ist","time_ist","condition_text","description"]

def make_xy(f_df):
    tr = f_df.drop(columns=[c for c in BASE_DROP if c in f_df.columns])
    tr = pd.get_dummies(tr, columns=["location"])
    tr = tr.select_dtypes(include=[np.number])
    mask = tr.notna().all(axis=1) & f_df["target"].notna()
    return tr.loc[mask], f_df.loc[mask,"target"]

def model_factory():
    """Returns a fresh instance each call (needed for LOSO folds)."""
    return VotingRegressor(
        estimators=[
            ("lr",  LinearRegression()),
            ("rf",  RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)),
            ("lgb", lgb.LGBMRegressor(n_estimators=200, n_jobs=-1, verbose=-1)),
        ], n_jobs=-1)

In [4]:
def leave_one_station_out(f_df, factory=model_factory, name="Voting"):
    """Train on 5 stations, predict held-out 6th; pool predictions across all folds."""
    stations = sorted(f_df.location.unique())
    all_true, all_pred = [], []
    per_station = {}

    for held in stations:
        train = f_df[f_df.location != held]
        test  = f_df[f_df.location == held]
        Xtr, ytr = make_xy(train)
        Xte, yte = make_xy(test)
        Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)

        m = factory()
        m.fit(Xtr, ytr)
        pred = m.predict(Xte)

        per_station[held] = dict(
            rmse=round(float(np.sqrt(((pred - yte.values)**2).mean())), 2),
            mae =round(float(np.abs(pred - yte.values).mean()), 2),
            n_test=len(yte),
        )
        all_true.append(yte.values)
        all_pred.append(pred)
        print(f"  {held:18s} n={len(yte):5d}  RMSE={per_station[held]['rmse']:6.2f}")

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    log_result(name, "T3", "B_leave_station_out", "full", "onehot",
               y_true, y_pred, n_train=None, notes=str(per_station))
    return y_true, y_pred, per_station

print("=== Protocol B: Voting ensemble, leave-one-station-out ===")
y_true_b, y_pred_b, per_stat = leave_one_station_out(feat_reset)

=== Protocol B: Voting ensemble, leave-one-station-out ===
  Anand Vihar        n= 8445  RMSE=  4.16
  Connaught Place    n= 8445  RMSE=  3.93
  Dwarka             n= 6724  RMSE=  5.54
  IGI Airport        n= 6724  RMSE=  5.39
  Okhla Phase III    n= 6724  RMSE=  5.86
  Rohini             n= 8445  RMSE=  4.08
[T3          |B_leave_station_out|full                  ] Voting                 RMSE=   4.80 MAE=  2.37 R2= 0.9955


In [5]:
def factory_xgb():
    return XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                        random_state=42, n_jobs=-1)

print("=== Protocol B: XGBoost, leave-one-station-out ===")
y_true_bx, y_pred_bx, per_stat_x = leave_one_station_out(feat_reset, factory_xgb, name="XGBoost")

=== Protocol B: XGBoost, leave-one-station-out ===
  Anand Vihar        n= 8445  RMSE=  3.96
  Connaught Place    n= 8445  RMSE=  3.93
  Dwarka             n= 6724  RMSE=  4.12
  IGI Airport        n= 6724  RMSE=  3.76
  Okhla Phase III    n= 6724  RMSE=  4.57
  Rohini             n= 8445  RMSE=  3.86
[T3          |B_leave_station_out|full                  ] XGBoost                RMSE=   4.03 MAE=  2.33 R2= 0.9968


In [6]:
def build_horizon_features(df, horizon):
    """target = aqi_index at t+horizon. All features are t-1 or earlier."""
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        for L in (1,2,3,6,12,24): g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        for W in (6,24):
            g[f"aqi_roll{W}_mean"]=g.aqi_index.shift(1).rolling(W).mean()
            g[f"aqi_roll{W}_max"] =g.aqi_index.shift(1).rolling(W).max()
        g["aqi_ewm12"]=g.aqi_index.shift(1).ewm(span=12).mean()
        g["aqi_mom3"]=g.aqi_lag1-g.aqi_lag3
        t=g.index.get_level_values("dt")
        g["hour"]=t.hour; g["month"]=t.month; g["dow"]=t.dayofweek
        g["hour_sin"]=np.sin(2*np.pi*t.hour/24); g["hour_cos"]=np.cos(2*np.pi*t.hour/24)
        g["month_sin"]=np.sin(2*np.pi*t.month/12); g["month_cos"]=np.cos(2*np.pi*t.month/12)
        g["target"] = g.aqi_index.shift(-horizon)
        g = g.drop(columns=["aqi_index","pm2_5","pm10","co","no2"])
        parts.append(g)
    return pd.concat(parts).dropna(subset=["target","aqi_lag24"])

CUTOFF = pd.Timestamp("2025-10-15")

print("=== Multi-horizon forecasts (XGBoost) ===")
for h in (1, 6, 24):
    f = build_horizon_features(clean, h).reset_index()
    tr = f[f.dt < CUTOFF]; te = f[f.dt >= CUTOFF]
    Xtr, ytr = make_xy(tr); Xte, yte = make_xy(te)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    m = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                     random_state=42, n_jobs=-1).fit(Xtr, ytr)
    log_result(f"XGB[h={h}h]", "T2", f"A_temporal_h{h}", "full", "onehot",
               yte, m.predict(Xte), n_train=len(Xtr))

=== Multi-horizon forecasts (XGBoost) ===
[T2          |A_temporal_h1 |full                  ] XGB[h=1h]              RMSE=   3.93 MAE=  2.34 R2= 0.9844
[T2          |A_temporal_h6 |full                  ] XGB[h=6h]              RMSE=  13.64 MAE=  8.52 R2= 0.8128
[T2          |A_temporal_h24|full                  ] XGB[h=24h]             RMSE=  36.01 MAE= 24.95 R2=-0.2963


In [7]:
# Paired test: Voting vs XGBoost-tuned on Protocol A residuals
from sklearn.model_selection import train_test_split

f = build_horizon_features(clean, 1).reset_index()
tr = f[f.dt < CUTOFF]; te = f[f.dt >= CUTOFF]
Xtr, ytr = make_xy(tr); Xte, yte = make_xy(te)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)

vot = VotingRegressor(estimators=[
    ("lr",  LinearRegression()),
    ("rf",  RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)),
    ("lgb", lgb.LGBMRegressor(n_estimators=200, n_jobs=-1, verbose=-1)),
], n_jobs=-1).fit(Xtr, ytr)

xgb = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                   random_state=42, n_jobs=-1).fit(Xtr, ytr)

resid_v = (vot.predict(Xte) - yte.values)**2
resid_x = (xgb.predict(Xte) - yte.values)**2
 
stat, p = wilcoxon(resid_v, resid_x)
print(f"Wilcoxon test (Voting vs XGBoost):")
print(f"  statistic = {stat:.2f}")
print(f"  p-value   = {p:.6f}")
print(f"  Voting RMSE  = {np.sqrt(resid_v.mean()):.2f}")
print(f"  XGBoost RMSE = {np.sqrt(resid_x.mean()):.2f}")
if p < 0.05:
    print("  ✅ Voting is significantly better than XGBoost (p<0.05)")
else:
    print("  ⚠️ Difference is NOT significant — do not claim Voting wins in the paper")

Wilcoxon test (Voting vs XGBoost):
  statistic = 30602020.00
  p-value   = 0.008288
  Voting RMSE  = 3.65
  XGBoost RMSE = 3.93
  ✅ Voting is significantly better than XGBoost (p<0.05)


In [8]:
tbl = pd.DataFrame(RESULTS).sort_values("rmse").reset_index(drop=True)
display(tbl)
tbl.to_csv(OUT/"results_protocolB.csv", index=False)
print(f"\nsaved → outputs/results_protocolB.csv  ({len(tbl)} rows)")

,model,tier,protocol,features,encoding,rmse,mae,r2,n_train,n_test,notes
0,XGB[h=1h],T2,A_temporal_h1,full,onehot,3.93,2.34,0.9844,34281.0,11226,
1,XGBoost,T3,B_leave_station_out,full,onehot,4.03,2.33,0.9968,NaN,45507,"{'Anand Vihar': {'rmse': 3.96, 'mae': 2.33, 'n..."
2,Voting,T3,B_leave_station_out,full,onehot,4.80,2.37,0.9955,NaN,45507,"{'Anand Vihar': {'rmse': 4.16, 'mae': 2.24, 'n..."
3,XGB[h=6h],T2,A_temporal_h6,full,onehot,13.64,8.52,0.8128,34281.0,11196,
4,XGB[h=24h],T2,A_temporal_h24,full,onehot,36.01,24.95,-0.2963,34281.0,11088,



saved → outputs/results_protocolB.csv  (5 rows)


In [9]:
# Verify 24h setup: how many test rows, what does persistence get at 24h?
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np, pandas as pd
from pathlib import Path

clean = pd.read_pickle(Path("outputs/clean.pkl"))
CUTOFF = pd.Timestamp("2025-10-15")

# Build 24h-ahead target with correct lags: use aqi_lag24 as the strongest, not aqi_lag1
f = build_horizon_features(clean, 24).reset_index()
tr = f[f.dt < CUTOFF]; te = f[f.dt >= CUTOFF]
print(f"24h — train rows: {len(tr)}, test rows: {len(te)}")

# Baseline: persistence at 24h means predicting tomorrow = today's aqi_lag24
persist_rmse = np.sqrt(((te.aqi_lag24 - te.target)**2).mean())
persist_r2   = r2_score(te.target, te.aqi_lag24)
print(f"24h PERSISTENCE  RMSE={persist_rmse:.2f}  R²={persist_r2:.4f}")

# Compare with model
Xtr, ytr = make_xy(tr); Xte, yte = make_xy(te)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
m = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                 random_state=42, n_jobs=-1).fit(Xtr, ytr)
pred = m.predict(Xte)
print(f"24h XGBoost      RMSE={np.sqrt(((pred - yte.values)**2).mean()):.2f}  R²={r2_score(yte, pred):.4f}")

# What if we remove the short-horizon lags (keep only lag24)?
drop_short = [c for c in Xte.columns if c.startswith(("aqi_lag1","aqi_lag2","aqi_lag3","aqi_lag6","aqi_lag12"))
              or c.startswith(("aqi_roll6","aqi_mom3"))]
Xtr_l = Xtr.drop(columns=[c for c in drop_short if c in Xtr.columns])
Xte_l = Xte.drop(columns=[c for c in drop_short if c in Xte.columns])
m2 = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                  random_state=42, n_jobs=-1).fit(Xtr_l, ytr)
pred2 = m2.predict(Xte_l)
print(f"24h XGB[lag24 only] RMSE={np.sqrt(((pred2 - yte.values)**2).mean()):.2f}  R²={r2_score(yte, pred2):.4f}")

24h — train rows: 34281, test rows: 11088
24h PERSISTENCE  RMSE=37.04  R²=-0.3708
24h XGBoost      RMSE=36.01  R²=-0.2963
24h XGB[lag24 only] RMSE=41.94  R²=-0.7576


In [10]:
# ---- 24H BASELINE COMPARISON ----
import numpy as np
from sklearn.metrics import r2_score

print("24h forecast — baseline comparison on test set:")
for name, pred in [
    ("Climatology (train mean)", np.full(len(te), tr.target.mean())),
    ("aqi_lag24  (yesterday's hour)", te.aqi_lag24.values),
    ("aqi_roll24_mean  (last-24h average)", te.aqi_roll24_mean.values),
    ("XGBoost (full features)", pred),
    ("XGBoost (lag24 only)", pred2),
]:
    rmse = float(np.sqrt(((pred - te.target.values)**2).mean()))
    r2   = float(r2_score(te.target, pred))
    mae  = float(np.abs(pred - te.target.values).mean())
    print(f"  {name:38s} RMSE={rmse:6.2f}  MAE={mae:5.2f}  R²={r2:+.4f}")

24h forecast — baseline comparison on test set:
  Climatology (train mean)               RMSE= 31.70  MAE=22.83  R²=-0.0041
  aqi_lag24  (yesterday's hour)          RMSE= 37.04  MAE=25.00  R²=-0.3708
  aqi_roll24_mean  (last-24h average)    RMSE= 31.16  MAE=20.74  R²=+0.0295
  XGBoost (full features)                RMSE= 36.01  MAE=24.95  R²=-0.2963
  XGBoost (lag24 only)                   RMSE= 41.94  MAE=28.42  R²=-0.7576


In [11]:
# ---- 24H RESULTS (final numbers for the paper) ----
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.metrics import r2_score

OUT = Path("outputs")
rows_24h = [
    {"model":"Climatology (train mean)",    "rmse":31.70, "mae":22.83, "r2":-0.0041},
    {"model":"Persistence (aqi_lag24)",     "rmse":37.04, "mae":25.00, "r2":-0.3708},
    {"model":"Persistence (aqi_roll24_mean)","rmse":31.16,"mae":20.74, "r2":+0.0295},
    {"model":"XGBoost (full features)",     "rmse":36.01, "mae":24.95, "r2":-0.2963},
    {"model":"XGBoost (lag24 only)",        "rmse":41.94, "mae":28.42, "r2":-0.7576},
]
df_24h = pd.DataFrame(rows_24h)
df_24h.to_csv(OUT/"results_24h.csv", index=False)
print(df_24h.to_string(index=False))
print("\nsaved → outputs/results_24h.csv")

                        model  rmse   mae      r2
     Climatology (train mean) 31.70 22.83 -0.0041
      Persistence (aqi_lag24) 37.04 25.00 -0.3708
Persistence (aqi_roll24_mean) 31.16 20.74  0.0295
      XGBoost (full features) 36.01 24.95 -0.2963
         XGBoost (lag24 only) 41.94 28.42 -0.7576

saved → outputs/results_24h.csv


In [12]:
# ---- OPTIONAL: 24H with trend feature ----
# Test if adding "is the last-24h avg rising or falling?" helps

def build_24h_with_trend(df):
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        g["aqi_lag24"]     = g.aqi_index.shift(24)
        g["aqi_roll24_mean"]= g.aqi_index.shift(1).rolling(24).mean()
        g["aqi_roll24_trend"]=(g.aqi_index.shift(1).rolling(6).mean()
                              -g.aqi_index.shift(1).rolling(24).mean())
        g["aqi_roll48_mean"]= g.aqi_index.shift(1).rolling(48).mean()
        t = g.index.get_level_values("dt")
        g["hour"]=t.hour; g["month"]=t.month
        g["hour_sin"]=np.sin(2*np.pi*t.hour/24); g["hour_cos"]=np.cos(2*np.pi*t.hour/24)
        g["month_sin"]=np.sin(2*np.pi*t.month/12); g["month_cos"]=np.cos(2*np.pi*t.month/12)
        g["temp_c"]=g.temp_c; g["humidity"]=g.humidity; g["windspeed_kph"]=g.windspeed_kph
        g["target"] = g.aqi_index.shift(-24)
        g = g.drop(columns=["aqi_index","pm2_5","pm10","co","no2"])
        parts.append(g)
    return pd.concat(parts).dropna(subset=["target","aqi_roll48_mean"])

from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error

f = build_24h_with_trend(clean).reset_index()
tr = f[f.dt < CUTOFF]; te = f[f.dt >= CUTOFF]
drop = ["target","date_ist","time_ist","condition_text","description"]
Xtr = pd.get_dummies(tr.drop(columns=[c for c in drop if c in tr.columns]), columns=["location"])
Xte = pd.get_dummies(te.drop(columns=[c for c in drop if c in te.columns]), columns=["location"])
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0).select_dtypes(include=[np.number])
Xtr = Xtr.select_dtypes(include=[np.number])

for name, m in [
    ("XGBoost [trend-features]", XGBRegressor(n_estimators=400, max_depth=6, n_jobs=-1, random_state=42)),
    ("Ridge [trend-features]",   Ridge()),
]:
    m.fit(Xtr, tr.target)
    p = m.predict(Xte)
    print(f"  {name:28s} RMSE={np.sqrt(((p-te.target)**2).mean()):6.2f}  R²={r2_score(te.target, p):+.4f}")

  XGBoost [trend-features]     RMSE= 47.56  R²=-1.2606
  Ridge [trend-features]       RMSE= 38.28  R²=-0.4648


In [13]:
# ---- FINAL SNAPSHOT: save everything before NB04 ----
import pandas as pd, json
from pathlib import Path

OUT = Path("outputs")

# Consolidate all results into one master file
all_results = []
for fname in ["results_v2_all_models.csv",   # NB01+NB02
              "results_protocolB.csv",       # NB03 (LOSO + horizons)
              "results_24h.csv"]:            # NB03 24h table
    p = OUT/fname
    if p.exists():
        all_results.append(pd.read_csv(p))
        print(f"loaded {fname}: {len(pd.read_csv(p))} rows")
    else:
        print(f"⚠️ {fname} missing")

master = pd.concat(all_results, ignore_index=True)
master.to_csv(OUT/"results_master.csv", index=False)
print(f"\nsaved outputs/results_master.csv ({len(master)} rows)")

# Summary file for the paper's Discussion
summary = {
    "scenario_A_churn":   {"best_model":"Tuned XGBoost", "metric":"84.55% accuracy"},
    "scenario_B_aqi_1h":  {"best_model":"Voting(LR+RF+LGBM)", "rmse":3.41, "r2":0.9883},
    "scenario_B_aqi_6h":  {"best_model":"XGBoost", "rmse":13.64, "r2":0.8128},
    "scenario_B_aqi_24h": {"best_model":"aqi_roll24_mean", "rmse":31.16, "r2":0.0295,
                            "verdict":"infeasible without external meteorological data"},
    "scenario_B_loso":    {"best_model":"XGBoost", "rmse":4.03, "r2":0.9968,
                            "finding":"ranking inverts vs temporal holdout"},
    "baseline_persistence_1h": {"rmse":3.90, "improvement_by_best": "12.6%"},
}
(OUT/"paper_summary.json").write_text(json.dumps(summary, indent=2))
print("saved outputs/paper_summary.json")

loaded results_v2_all_models.csv: 26 rows
loaded results_protocolB.csv: 5 rows
loaded results_24h.csv: 5 rows

saved outputs/results_master.csv (36 rows)
saved outputs/paper_summary.json


In [14]:
# ---- SAVE 6H MODEL FOR APP ----
import pickle, pandas as pd, numpy as np
from pathlib import Path
from xgboost import XGBRegressor

OUT = Path("outputs")
clean = pd.read_pickle(OUT/"clean.pkl")
CUTOFF = pd.Timestamp("2025-10-15")

def build_6h_features(df):
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        for L in (1,2,3,6,12,24): g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        for W in (6,24):
            g[f"aqi_roll{W}_mean"]=g.aqi_index.shift(1).rolling(W).mean()
            g[f"aqi_roll{W}_max"] =g.aqi_index.shift(1).rolling(W).max()
        g["aqi_ewm12"]=g.aqi_index.shift(1).ewm(span=12).mean()
        g["aqi_mom3"]=g.aqi_lag1-g.aqi_lag3
        t=g.index.get_level_values("dt")
        g["hour"]=t.hour; g["month"]=t.month; g["dow"]=t.dayofweek
        g["hour_sin"]=np.sin(2*np.pi*t.hour/24); g["hour_cos"]=np.cos(2*np.pi*t.hour/24)
        g["month_sin"]=np.sin(2*np.pi*t.month/12); g["month_cos"]=np.cos(2*np.pi*t.month/12)
        g["pressure_delta"]=g.pressure_mb.diff()
        g["wind_x_lag1"]=g.windspeed_kph*g.aqi_lag1
        g["target"]=g.aqi_index.shift(-6)
        g = g.drop(columns=["aqi_index","pm2_5","pm10","co","no2"])
        parts.append(g)
    return pd.concat(parts).dropna(subset=["target","aqi_lag24"])

f = build_6h_features(clean).reset_index()
tr = f[f.dt < CUTOFF]
drop = ["target","date_ist","time_ist","condition_text","description"]
Xtr = pd.get_dummies(tr.drop(columns=[c for c in drop if c in tr.columns]), columns=["location"])
Xtr = Xtr.select_dtypes(include=[np.number])

m6 = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                  random_state=42, n_jobs=-1).fit(Xtr, tr.target)

pickle.dump({"model": m6, "columns": list(Xtr.columns)}, open(OUT/"best_model_6h.pkl","wb"))
print(f"saved outputs/best_model_6h.pkl with {len(Xtr.columns)} features")

saved outputs/best_model_6h.pkl with 28 features
